In [ ]:
import re
from pathlib import Path

import emcee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

import bay
from bay import IDX

RESULTS = Path("results")
N_TAU_MIN = 50
Q_ERR_MAX = 0.1                                          # posterior sigma
QUANTILES = [0.05, 0.16, 0.5, 0.84, 0.95]
CORE, TAILS = [1, 2, 3], [0, 4]                          # 16/50/84 gate, 5/95 reported
LENGTHS = [2500, 5000, 10000, 15000, 20000, 30000, 40000, 50000]   # steps after burn-in
SAMPLER = ["nwalkers", "nburn", "restart_keep", "restart_jitter", "moves"]
INK, MUTED, GRID = "#52514e", "#898781", "#e1e0d9"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]    # fixed order, never cycled


def latest(notes):
    runs = sorted(RESULTS.glob(f"*_{notes}"))
    if not runs:
        raise FileNotFoundError(f"no {RESULTS}/*_{notes} run yet: bay.run it from main.ipynb first")
    return runs[-1]


def load_run(run_dir):
    chains = {}
    for path in sorted(Path(run_dir).glob("chain_*.npz")):
        c = bay.load(path)
        chains[c["spec"]["name"]] = c
    return chains


def config(name):
    return re.sub(r"_s\d+$", "", name)


def quantities(x):
    """(..., 12) samples -> the physical quantities we compare."""
    h = x[..., IDX["sqrt_eccentricity_cos_periastron_argument"]]
    q = x[..., IDX["sqrt_eccentricity_sin_periastron_argument"]]
    return {"P_d": np.exp(x[..., IDX["ln_orbital_period"]]),
            "e": h * h + q * q,
            "k": np.exp(x[..., IDX["ln_planet_to_star_radius_ratio"]]),
            "b": x[..., IDX["impact_parameter"]],
            "rho": 10 ** x[..., IDX["log10_stellar_density"]]}


def head(c, n):
    """First n steps after burn-in, (n / thin, nwalkers, 12)."""
    return c["chain"][: n // c["spec"]["thin"]].astype(float)


def tau_max(c):
    return emcee.autocorr.integrated_time(c["chain"].astype(float), tol=0).max() * c["spec"]["thin"]


def chain_quantiles(c, n):
    flat = head(c, n).reshape(-1, len(bay.NAMES))
    return {k: np.quantile(v, QUANTILES) for k, v in quantities(flat).items()}


def reference(chains):
    """Median over chains of each full-length quantile, and of each posterior sigma."""
    full = [(c, c["spec"]["nsteps"]) for c in chains.values()]
    qs = [chain_quantiles(c, n) for c, n in full]
    sig = [{k: v.std() for k, v in quantities(head(c, n).reshape(-1, len(bay.NAMES))).items()}
           for c, n in full]
    return ({k: np.median([q[k] for q in qs], axis=0) for k in qs[0]},
            {k: np.median([s[k] for s in sig]) for k in sig[0]})


def q_err(c, n, ref, which=CORE):
    q_ref, sigma = ref
    return max(np.abs(q - q_ref[k])[which].max() / sigma[k] for k, q in chain_quantiles(c, n).items())


def scan(chains, ref):
    """One row per chain and truncation length."""
    rows = []
    for name, c in chains.items():
        s, t = c["spec"], tau_max(c)
        for n in [n for n in LENGTHS if n < s["nsteps"]] + [s["nsteps"]]:
            rows.append(dict(chain=name, config=config(name), seed=s["seed"], n=n,
                             tau=t, n_tau=n / t, q_err=q_err(c, n, ref), tail_err=q_err(c, n, ref, TAILS),
                             cost=s["nwalkers"] * (s["nburn"] + n)))
    df = pd.DataFrame(rows)
    df["viable"] = (df.n_tau >= N_TAU_MIN) & (df.q_err <= Q_ERR_MAX)
    return df


def varied(spec, base):
    return ", ".join(f"{k}={spec[k]}" for k in SAMPLER if spec[k] != base[k]) or "base"


def style(ax):
    ax.grid(color=GRID, lw=0.6)
    ax.tick_params(colors=MUTED, labelcolor=INK)
    for side in ax.spines.values():
        side.set_color(GRID)

In [ ]:
tuning = load_run(latest("mcmc-tuning"))
ref = reference(tuning)
df = scan(tuning, ref)
base_spec = tuning["base_s0"]["spec"]
print(len(tuning), "chains;  reference median / sigma:")
pd.DataFrame({k: dict(median=ref[0][k][2], sigma=ref[1][k]) for k in ref[0]}).round(4)

In [ ]:
last = df[df.n == df.groupby("chain").n.transform("max")]
summary = last.groupby("config").agg(n=("n", "max"), tau=("tau", "max"), n_tau=("n_tau", "min"),
                                     q_err=("q_err", "max"), tail_err=("tail_err", "max"),
                                     viable=("viable", "all"))
first = {config(k): c for k, c in sorted(tuning.items())}
summary.insert(0, "varied", [varied(first[k]["spec"], base_spec) for k in summary.index])
summary["acc"] = [np.mean([c["acc"].mean() for k, c in tuning.items() if config(k) == cfg])
                  for cfg in summary.index]
summary["minutes"] = [np.mean([float(c["seconds"]) for k, c in tuning.items() if config(k) == cfg]) / 60
                      for cfg in summary.index]
summary.sort_values("tau").round(3)

In [ ]:
FACTORS = {"moves": None, "walkers": "nwalkers", "burn": "nburn", "keep": "restart_keep", "jitter": "restart_jitter"}
worst = df.groupby(["config", "n"]).agg(n_tau=("n_tau", "min"), q_err=("q_err", "max"),
                                        cost=("cost", "max")).reset_index()

fig, axes = plt.subplots(2, len(FACTORS), figsize=(4 * len(FACTORS), 7), sharey="row", sharex="row")
for j, (factor, key) in enumerate(FACTORS.items()):
    members = ["base"] + sorted(c for c in worst.config.unique() if c.startswith(factor))
    for i, cfg in enumerate(members):
        d = worst[worst.config == cfg]
        kw = dict(color=MUTED if cfg == "base" else SERIES[i - 1], lw=2, marker="o", ms=4,
                  label=cfg if cfg == "base" else first[cfg]["spec"][key] if key else cfg[len(factor):].strip("_"))
        axes[0, j].plot(d.n, d.n_tau, **kw)
        axes[1, j].plot(d.cost / 1e6, d.q_err, **kw)
    axes[0, j].axhline(N_TAU_MIN, color=INK, lw=1, ls="--")
    axes[1, j].axhline(Q_ERR_MAX, color=INK, lw=1, ls="--")
    axes[0, j].set_title(factor, color=INK)
    axes[0, j].legend(frameon=False, fontsize=8)
    axes[0, j].set_xlabel("steps after burn-in")
    axes[1, j].set_xlabel("likelihood calls [M]")
    for ax in axes[:, j]:
        style(ax)
axes[0, 0].set_ylabel("n / tau_max  (worst seed)")
axes[1, 0].set_ylabel("q_err [sigma]  (worst seed)")
axes[1, 0].set_yscale("log")
fig.tight_layout()

In [ ]:
configs = sorted({config(k) for k in tuning}, key=lambda c: (c != "base", c))
ncol = 3
fig, axes = plt.subplots(-(-len(configs) // ncol), ncol, figsize=(4 * ncol, 2.6 * -(-len(configs) // ncol)),
                         sharey=True, squeeze=False)
for ax, cfg in zip(axes.flat, configs):
    for i, (name, c) in enumerate(sorted((k, c) for k, c in tuning.items() if config(k) == cfg)):
        lnp = c["chain"][:, :, IDX["ln_orbital_period"]]
        steps = np.arange(lnp.shape[0]) * c["spec"]["thin"]
        lo, mid, hi = np.quantile(lnp, [0.16, 0.5, 0.84], axis=1)
        ax.fill_between(steps, lo, hi, color=SERIES[i], alpha=0.2, lw=0)
        ax.plot(steps, mid, color=SERIES[i], lw=1, label=f"seed {c['spec']['seed']}")
    ax.set_title(cfg, color=INK, fontsize=9)
    style(ax)
for ax in axes.flat[len(configs):]:
    ax.remove()
axes[0, 0].set_ylabel("ln P")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()

In [ ]:
def min_viable(d):
    ok = d.groupby("n").viable.all().sort_index()
    bad = ok.index[~ok.values]
    later = ok.index[ok.index > bad.max()] if len(bad) else ok.index
    return later.min() if len(later) else np.nan

pick = pd.DataFrame({cfg: dict(n_min=min_viable(d)) for cfg, d in df.groupby("config")}).T
spec_of = {cfg: first[cfg]["spec"] for cfg in pick.index}
pick["cost_M"] = [spec_of[c]["nwalkers"] * (spec_of[c]["nburn"] + n) / 1e6 for c, n in pick.n_min.items()]
pick.insert(0, "varied", [varied(spec_of[c], base_spec) for c in pick.index])
pick = pick.sort_values("cost_M")
winner = pick.cost_M.idxmin() if pick.cost_M.notna().any() else None
print("winner:", winner, "" if winner is None else f"n_min = {int(pick.n_min[winner])}")
pick.round(2)

In [ ]:
WRITE = False

if winner is not None:
    s = dict(spec_of[winner], nsteps=int(pick.n_min[winner]))
    s["moves"] = "{" + ", ".join(f"{k}: {v}" for k, v in s["moves"].items()) + "}"
    path = Path("seed_consistency.yaml")
    text = path.read_text(encoding="utf-8")
    for key in SAMPLER + ["nsteps"]:
        text = re.sub(rf"(?m)^(    {key}: )[^#\n]*", rf"\g<1>{s[key]}", text)
    text = re.sub(r"(?s)# PLACEHOLDER.*?running\.\n", f"# Config: {winner} from the tuning run, n_min = {s['nsteps']}.\n", text)
    print(text[: text.index("  - name:", text.index("  - name:") + 1)])
    if WRITE:
        path.write_text(text, encoding="utf-8")
        print("written", path)

In [ ]:
cons = load_run(latest("seed-consistency"))
cons_ref = reference(cons)
cdf = scan(cons, cons_ref)
full = cdf[cdf.n == cdf.groupby("chain").n.transform("max")].set_index("chain")
full["acc"] = [cons[k]["acc"].mean() for k in full.index]
print(len(cons), "chains,", varied(next(iter(cons.values()))["spec"], base_spec), ";",
      f"{full.viable.mean():.0%} of seeds viable at full length")
full[["seed", "n", "acc", "tau", "n_tau", "q_err", "tail_err", "viable"]].round(3)

In [ ]:
def rhat(samples):
    """samples: (m chains, N). Classic Gelman-Rubin."""
    N = samples.shape[1]
    W = samples.var(axis=1, ddof=1).mean()
    B_over_N = samples.mean(axis=1).var(ddof=1)
    return np.sqrt(((N - 1) / N * W + B_over_N) / W)

per_seed = {k: quantities(c["chain"].astype(float).reshape(-1, len(bay.NAMES))) for k, c in cons.items()}
stats = {}
for q in cons_ref[0]:
    meds = np.array([np.median(v[q]) for v in per_seed.values()])
    stats[q] = dict(median=cons_ref[0][q][2], sigma=cons_ref[1][q],
                    median_scatter_sigma=meds.std(ddof=1) / cons_ref[1][q],
                    rhat=rhat(np.array([v[q] for v in per_seed.values()])))
pd.DataFrame(stats).T.round(4)

In [ ]:
fig, axes = plt.subplots(1, len(cons_ref[0]), figsize=(3.2 * len(cons_ref[0]), 3.6), sharey=True)
seeds = [cons[k]["spec"]["seed"] for k in per_seed]
for ax, q in zip(axes, cons_ref[0]):
    lo, mid, hi = np.array([np.quantile(v[q], [0.16, 0.5, 0.84]) for v in per_seed.values()]).T
    ref_lo, ref_mid, ref_hi = cons_ref[0][q][[1, 2, 3]]
    ax.axvspan(ref_lo, ref_hi, color=GRID, lw=0)
    ax.axvline(ref_mid, color=MUTED, lw=1)
    ax.errorbar(mid, np.arange(len(seeds)), xerr=[mid - lo, hi - mid], fmt="o", ms=5,
                color=SERIES[0], ecolor=SERIES[0], elinewidth=2, capsize=0)
    ax.set_title(q, color=INK)
    style(ax)
axes[0].set_yticks(range(len(seeds)), [f"seed {s}" for s in seeds])
fig.suptitle("median and 16-84% per seed; band = reference", color=INK)
fig.tight_layout()